# DA5401W Model Prediction & Interpretation using SHAP Analysis


## Section 1: Why Model Interpretability Matters

**Key Advantages:**
- ✓ Theoretically sound (based on game theory)
- ✓ Works with any model type
- ✓ Provides both local and global explanations
- ✓ Identifies feature interactions
- ✓ Detects model biases


## Section 2: SHAP Theory (Simplified)



## Section 3: Dataset & Model Setup

We'll use the same Breast Cancer dataset and train a linear model (Logistic Regression) for interpretability.

In [ ]:
# Import essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries imported successfully!")
print(f"SHAP version: {shap.__version__}")

In [ ]:
# Load and prepare the dataset
cancer_data = load_breast_cancer()
X = cancer_data.data
y = cancer_data.target
feature_names = cancer_data.feature_names
target_names = cancer_data.target_names

# Create a DataFrame
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
df['target_name'] = df['target'].map({0: target_names[0], 1: target_names[1]})

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nClass distribution:")
print(df['target_name'].value_counts())

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train_scaled.shape[0]}")
print(f"Test set size: {X_test_scaled.shape[0]}")
print(f"Number of features: {X_train_scaled.shape[1]}")

In [ ]:
# Train a Logistic Regression Model
# Linear models are ideal for SHAP learning because:
# 1. Exact SHAP calculations are efficient
# 2. Results are easily interpretable
# 3. Coefficients have direct meaning

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate model
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print("Logistic Regression Model Trained!")
print(f"\nTest Accuracy: {accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

In [ ]:
# Model Coefficients (Traditional Interpretation)
print("Model Coefficients (Traditional Linear Model Interpretation):")
print("="*70)

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print(coef_df.to_string(index=False))
print(f"\nIntercept (Base): {model.intercept_[0]:.4f}")

## Section 4: SHAP for Linear Models

### Why Linear Models?

Linear models are perfect for learning SHAP because:
1. **Exact SHAP calculations**: SHAP values can be computed exactly and efficiently
2. **Interpretability**: Results align with model coefficients
3. **Transparency**: Easy to understand feature contributions

### SHAP vs. Coefficients

- **Coefficients**: Global feature importance (same for all predictions)
- **SHAP values**: Local feature importance (varies per prediction)

For linear models, SHAP values are proportional to coefficients, but SHAP provides additional insights about individual predictions.


In [ ]:
# Create SHAP Explainer for Linear Model
# For linear models, SHAP uses exact calculations

explainer = shap.LinearExplainer(model, X_train_scaled)
shap_values = explainer.shap_values(X_test_scaled)

print("SHAP Explainer created for Logistic Regression!")
print(f"\nSHAP values shape: {shap_values.shape}")
print(f"Number of test samples: {X_test_scaled.shape[0]}")
print(f"Number of features: {X_test_scaled.shape[1]}")
print(f"\nBase value (expected model output): {explainer.expected_value:.4f}")

In [ ]:
# Examine SHAP values for a few predictions
print("\nSHAP Values for First 3 Test Samples:")
print("="*80)

for i in range(3):
    print(f"\nSample {i+1}:")
    print(f"  Actual Class: {target_names[y_test[i]]}")
    print(f"  Predicted Probability: {model.predict_proba(X_test_scaled[i:i+1])[0][1]:.4f}")
    print(f"  Base Value: {explainer.expected_value:.4f}")
    print(f"  Sum of SHAP values: {shap_values[i].sum():.4f}")
    print(f"  Final Prediction: {explainer.expected_value + shap_values[i].sum():.4f}")
    print(f"\n  Top 5 Contributing Features:")
    
    # Get top contributing features
    top_indices = np.argsort(np.abs(shap_values[i]))[-5:][::-1]
    for rank, idx in enumerate(top_indices, 1):
        print(f"    {rank}. {feature_names[idx]}: SHAP = {shap_values[i][idx]:+.4f}")

In [ ]:
# Comparison: SHAP Values vs. Coefficients
print("\nComparison: SHAP Values vs. Model Coefficients")
print("="*80)

# Average absolute SHAP values
mean_abs_shap = np.abs(shap_values).mean(axis=0)

comparison_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_[0],
    'Mean |SHAP|': mean_abs_shap
}).sort_values('Mean |SHAP|', ascending=False)

print(comparison_df.head(10).to_string(index=False))
print("\nNote: Mean |SHAP| represents average absolute contribution across all predictions")
print("      Coefficients represent global feature importance")

## Section 5: Local Explanations (Individual Predictions)

Local explanations show how features contribute to a specific prediction. We'll use three visualization types:
1. **Force Plot**: Shows how features push prediction up/down
2. **Waterfall Plot**: Shows cumulative feature contributions
3. **Decision Plot**: Shows multiple predictions and their explanations


In [ ]:
# Force Plot for a Single Prediction
# This shows how each feature contributes to pushing the prediction away from the base value

sample_idx = 0
print(f"\nForce Plot for Sample {sample_idx}:")
print(f"Actual Class: {target_names[y_test[sample_idx]]}")
print(f"Predicted Probability: {model.predict_proba(X_test_scaled[sample_idx:sample_idx+1])[0][1]:.4f}")

# Create force plot
shap.force_plot(
    explainer.expected_value,
    shap_values[sample_idx],
    X_test_scaled[sample_idx],
    feature_names=feature_names,
    matplotlib=True,
    show=False
)
plt.tight_layout()
plt.show()

In [ ]:
# Waterfall Plot for a Single Prediction
# This shows cumulative feature contributions

sample_idx = 0
explanation = shap.Explanation(
    values=shap_values[sample_idx],
    base_values=explainer.expected_value,
    data=X_test_scaled[sample_idx],
    feature_names=feature_names
)

plt.figure(figsize=(12, 8))
shap.plots.waterfall(explanation, show=False)
plt.tight_layout()
plt.show()

print(f"\nWaterfall Plot Interpretation:")
print(f"Base Value: {explainer.expected_value:.4f}")
print(f"Final Prediction: {explainer.expected_value + shap_values[sample_idx].sum():.4f}")
print(f"Each feature either increases (red) or decreases (blue) the prediction")

In [ ]:
# Decision Plot for Multiple Predictions
# Shows how different samples' predictions are built up from the base value

plt.figure(figsize=(12, 8))
shap.decision_plot(
    explainer.expected_value,
    shap_values[:50],  # First 50 test samples
    X_test_scaled[:50],
    feature_names=feature_names,
    show=False
)
plt.tight_layout()
plt.show()

print("Decision Plot Interpretation:")
print("- Each line represents one prediction")
print("- Lines start at the base value (left)")
print("- Each feature shifts the line left (negative) or right (positive)")
print("- Final position is the model's prediction")

In [ ]:
# Case Study: Detailed Interpretation of Specific Predictions
print("\n" + "="*80)
print("CASE STUDY: Detailed Interpretation of Individual Predictions")
print("="*80)

# Find interesting cases: correct and incorrect predictions
y_pred_test = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Case 1: Correct prediction with high confidence
correct_high_conf = np.where((y_pred_test == y_test) & (y_pred_proba > 0.9))[0]
if len(correct_high_conf) > 0:
    idx = correct_high_conf[0]
    print(f"\nCase 1: Correct Prediction with High Confidence")
    print(f"  Actual: {target_names[y_test[idx]]}")
    print(f"  Predicted: {target_names[y_pred_test[idx]]} (Probability: {y_pred_proba[idx]:.4f})")
    print(f"  Base Value: {explainer.expected_value:.4f}")
    print(f"  Top 5 Contributing Features:")
    top_indices = np.argsort(np.abs(shap_values[idx]))[-5:][::-1]
    for rank, feat_idx in enumerate(top_indices, 1):
        print(f"    {rank}. {feature_names[feat_idx]}: SHAP = {shap_values[idx][feat_idx]:+.4f}")

# Case 2: Correct prediction with low confidence
correct_low_conf = np.where((y_pred_test == y_test) & (y_pred_proba < 0.6) & (y_pred_proba > 0.4))[0]
if len(correct_low_conf) > 0:
    idx = correct_low_conf[0]
    print(f"\nCase 2: Correct Prediction with Low Confidence (Borderline)")
    print(f"  Actual: {target_names[y_test[idx]]}")
    print(f"  Predicted: {target_names[y_pred_test[idx]]} (Probability: {y_pred_proba[idx]:.4f})")
    print(f"  Base Value: {explainer.expected_value:.4f}")
    print(f"  Top 5 Contributing Features:")
    top_indices = np.argsort(np.abs(shap_values[idx]))[-5:][::-1]
    for rank, feat_idx in enumerate(top_indices, 1):
        print(f"    {rank}. {feature_names[feat_idx]}: SHAP = {shap_values[idx][feat_idx]:+.4f}")

# Case 3: Incorrect prediction
incorrect = np.where(y_pred_test != y_test)[0]
if len(incorrect) > 0:
    idx = incorrect[0]
    print(f"\nCase 3: Incorrect Prediction (Misclassification)")
    print(f"  Actual: {target_names[y_test[idx]]}")
    print(f"  Predicted: {target_names[y_pred_test[idx]]} (Probability: {y_pred_proba[idx]:.4f})")
    print(f"  Base Value: {explainer.expected_value:.4f}")
    print(f"  Top 5 Contributing Features:")
    top_indices = np.argsort(np.abs(shap_values[idx]))[-5:][::-1]
    for rank, feat_idx in enumerate(top_indices, 1):
        print(f"    {rank}. {feature_names[feat_idx]}: SHAP = {shap_values[idx][feat_idx]:+.4f}")

## Section 6: Global Explanations (Model-Level Insights)

Global explanations show overall feature importance and how features affect predictions across the entire dataset.


In [ ]:
# Summary Plot: Overall Feature Importance
# Shows which features are most important for the model

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_test_scaled,
    feature_names=feature_names,
    plot_type='bar',
    show=False
)
plt.title('SHAP Summary Plot: Mean Absolute Feature Importance', fontsize=14, fontweight='bold')
plt.xlabel('Mean |SHAP value| (average impact on model output)', fontsize=11)
plt.tight_layout()
plt.show()

print("Summary Plot Interpretation:")
print("- Shows average absolute SHAP values for each feature")
print("- Higher values = more important features")
print("- Represents global feature importance")

In [ ]:
# Beeswarm Plot: Feature Values vs. SHAP Values
# Shows relationship between feature values and their SHAP values

plt.figure(figsize=(10, 10))
shap.summary_plot(
    shap_values,
    X_test_scaled,
    feature_names=feature_names,
    show=False
)
plt.title('SHAP Beeswarm Plot: Feature Values vs. SHAP Values', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Beeswarm Plot Interpretation:")
print("- Red points: High feature values")
print("- Blue points: Low feature values")
print("- Horizontal position: SHAP value (impact on prediction)")
print("- Shows how feature values correlate with their impact")

In [ ]:
# Dependence Plot: Feature Value vs. SHAP Value
# Shows how a specific feature's value affects its SHAP value

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Get top 4 important features
top_features_idx = np.argsort(np.abs(shap_values).mean(axis=0))[-4:][::-1]

for idx, (ax, feat_idx) in enumerate(zip(axes.flat, top_features_idx)):
    shap.dependence_plot(
        feat_idx,
        shap_values,
        X_test_scaled,
        feature_names=feature_names,
        ax=ax,
        show=False
    )
    ax.set_title(f'Dependence Plot: {feature_names[feat_idx]}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("Dependence Plot Interpretation:")
print("- X-axis: Feature value")
print("- Y-axis: SHAP value (impact on prediction)")
print("- Shows non-linear relationships between features and predictions")
print("- Color indicates interaction with another feature")

In [ ]:
# Feature Importance Ranking
print("\nGlobal Feature Importance (Mean Absolute SHAP Values):")
print("="*70)

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean |SHAP|': np.abs(shap_values).mean(axis=0),
    'Std SHAP': np.abs(shap_values).std(axis=0)
}).sort_values('Mean |SHAP|', ascending=False)

print(importance_df.head(15).to_string(index=False))

# Visualize
plt.figure(figsize=(10, 8))
plt.barh(range(len(importance_df.head(15))), importance_df.head(15)['Mean |SHAP|'], color='#3498db')
plt.yticks(range(len(importance_df.head(15))), importance_df.head(15)['Feature'])
plt.xlabel('Mean Absolute SHAP Value', fontsize=12, fontweight='bold')
plt.title('Top 15 Most Important Features', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Interaction Detection
# SHAP can identify which features interact with each other

print("\nFeature Interactions Analysis:")
print("="*70)

# Get the most important feature
most_important_idx = np.argsort(np.abs(shap_values).mean(axis=0))[-1]
most_important_feature = feature_names[most_important_idx]

print(f"\nAnalyzing interactions with: {most_important_feature}")
print(f"\nDependence plot shows color-coded interaction with another feature")

plt.figure(figsize=(12, 6))
shap.dependence_plot(
    most_important_idx,
    shap_values,
    X_test_scaled,
    feature_names=feature_names,
    show=False
)
plt.title(f'Feature Interaction: {most_important_feature}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nColor gradient indicates interaction strength with another feature")
print("Steeper color changes suggest stronger interactions")

## Section 7: Practical Applications & Limitations

### When to Use SHAP

✓ **High-stakes decisions**: Healthcare, finance, criminal justice  
✓ **Regulatory compliance**: GDPR, Fair Lending Laws require explainability  
✓ **Model debugging**: Understand why model makes mistakes  
✓ **Feature engineering**: Identify important features  
✓ **Bias detection**: Discover discriminatory patterns  

### SHAP vs. Other Methods

| Method | Pros | Cons |
|--------|------|------|
| **SHAP** | Theoretically sound, works with any model | Computationally expensive |
| **LIME** | Fast, model-agnostic | Less stable, local only |
| **Coefficients** | Fast, interpretable | Only for linear models |
| **Permutation Importance** | Fast, model-agnostic | Doesn't show direction |

### Computational Considerations

- **Linear models**: Exact SHAP calculation is fast
- **Tree models**: TreeExplainer is efficient
- **Deep learning**: KernelExplainer is slow (O(n²) complexity)
- **Large datasets**: Use sampling or approximations

### Limitations of SHAP

- ✗ Computationally expensive for large datasets
- ✗ Assumes feature independence (may not hold in practice)
- ✗ Can be misleading with highly correlated features
- ✗ Requires domain expertise to interpret correctly


In [ ]:
# Bias Detection using SHAP
# Identify if model predictions are biased towards certain feature values

print("\nBias Detection Analysis:")
print("="*70)

# Analyze SHAP values by prediction outcome
benign_predictions = y_pred_test == 1
malignant_predictions = y_pred_test == 0

print(f"\nAverage SHAP values for Benign predictions:")
benign_shap_mean = shap_values[benign_predictions].mean(axis=0)
top_benign = np.argsort(np.abs(benign_shap_mean))[-5:][::-1]
for rank, idx in enumerate(top_benign, 1):
    print(f"  {rank}. {feature_names[idx]}: {benign_shap_mean[idx]:+.4f}")

print(f"\nAverage SHAP values for Malignant predictions:")
malignant_shap_mean = shap_values[malignant_predictions].mean(axis=0)
top_malignant = np.argsort(np.abs(malignant_shap_mean))[-5:][::-1]
for rank, idx in enumerate(top_malignant, 1):
    print(f"  {rank}. {feature_names[idx]}: {malignant_shap_mean[idx]:+.4f}")

print("\n✓ Analysis complete. Check for consistent feature importance across classes.")

## Section 8: Hands-On Interpretation Challenge

Let's apply SHAP analysis to interpret model behavior and identify potential improvements.

In [ ]:
# Challenge 1: Explain a Misclassified Sample
print("\nCHALLENGE 1: Explain a Misclassified Sample")
print("="*70)

incorrect_idx = np.where(y_pred_test != y_test)[0]

if len(incorrect_idx) > 0:
    idx = incorrect_idx[0]
    print(f"\nMisclassified Sample Analysis:")
    print(f"  Actual Class: {target_names[y_test[idx]]}")
    print(f"  Predicted Class: {target_names[y_pred_test[idx]]}")
    print(f"  Prediction Confidence: {y_pred_proba[idx]:.4f}")
    print(f"\nFeatures pushing towards {target_names[y_pred_test[idx]]} (predicted):")
    
    if y_pred_test[idx] == 1:
        top_positive = np.argsort(shap_values[idx])[-3:][::-1]
        for rank, feat_idx in enumerate(top_positive, 1):
            print(f"  {rank}. {feature_names[feat_idx]}: SHAP = {shap_values[idx][feat_idx]:+.4f}, Value = {X_test_scaled[idx][feat_idx]:.4f}")
    else:
        top_negative = np.argsort(shap_values[idx])[:3]
        for rank, feat_idx in enumerate(top_negative, 1):
            print(f"  {rank}. {feature_names[feat_idx]}: SHAP = {shap_values[idx][feat_idx]:+.4f}, Value = {X_test_scaled[idx][feat_idx]:.4f}")
    
    print(f"\nInterpretation: The model made this mistake because...")
    print(f"(Your analysis here)")
else:
    print("\nNo misclassified samples found! Model achieved perfect accuracy on test set.")

In [ ]:
# Challenge 2: Identify Potential Model Biases
print("\nCHALLENGE 2: Identify Potential Model Biases")
print("="*70)

# Check if model relies heavily on a few features
feature_importance = np.abs(shap_values).mean(axis=0)
top_3_importance = np.sum(feature_importance[np.argsort(feature_importance)[-3:]])
total_importance = np.sum(feature_importance)
top_3_percentage = (top_3_importance / total_importance) * 100

print(f"\nFeature Concentration Analysis:")
print(f"  Top 3 features account for {top_3_percentage:.1f}% of total importance")

if top_3_percentage > 50:
    print(f"  ⚠️  WARNING: Model relies heavily on few features")
    print(f"  This could indicate:")
    print(f"    - Limited feature diversity")
    print(f"    - Potential overfitting")
    print(f"    - Need for feature engineering")
else:
    print(f"  ✓ Good: Model uses diverse features")

# Check for feature value extremes
print(f"\nFeature Value Distribution Analysis:")
for feat_idx in np.argsort(feature_importance)[-3:][::-1]:
    feat_values = X_test_scaled[:, feat_idx]
    print(f"  {feature_names[feat_idx]}:")
    print(f"    Min: {feat_values.min():.4f}, Max: {feat_values.max():.4f}, Mean: {feat_values.mean():.4f}")

In [ ]:
# Challenge 3: Suggest Model Improvements
print("\nCHALLENGE 3: Suggest Model Improvements")
print("="*70)

print(f"\nCurrent Model Performance:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Number of misclassifications: {np.sum(y_pred_test != y_test)}")

print(f"\nPotential Improvements:")
print(f"  1. Feature Engineering:")
print(f"     - Create interaction terms between top features")
print(f"     - Consider polynomial features for non-linear relationships")
print(f"\n  2. Data Collection:")
print(f"     - Collect more samples for underrepresented classes")
print(f"     - Ensure balanced representation of feature values")
print(f"\n  3. Model Tuning:")
print(f"     - Adjust class weights for imbalanced data")
print(f"     - Try regularization (L1/L2) to prevent overfitting")
print(f"\n  4. Feature Selection:")
print(f"     - Remove low-importance features")
print(f"     - Reduce multicollinearity")
print(f"\n  5. Ensemble Methods:")
print(f"     - Combine multiple models")
print(f"     - Use voting or stacking")

## Section 9: Summary and Best Practices

### Key Takeaways

1. **SHAP provides both local and global explanations**
   - Local: Why a specific prediction was made
   - Global: Which features are most important overall

2. **SHAP values are theoretically sound**
   - Based on Shapley values from game theory
   - Satisfy desirable properties (local accuracy, missingness, consistency)

3. **Linear models are ideal for learning SHAP**
   - Exact calculations are efficient
   - Results are easily interpretable
   - SHAP values align with model coefficients

4. **SHAP helps identify model issues**
   - Detect biases and unfair patterns
   - Understand misclassifications
   - Validate model behavior

### Best Practices for SHAP Analysis

1. **Start with global explanations** to understand overall model behavior
2. **Investigate local explanations** for interesting or problematic predictions
3. **Compare SHAP values** across different prediction outcomes
4. **Validate insights** with domain experts
5. **Use SHAP for model debugging** and improvement
6. **Document findings** for stakeholder communication

### SHAP in Production

- Use for model monitoring and drift detection
- Provide explanations to end users
- Detect and mitigate biases
- Support regulatory compliance
- Enable continuous model improvement


In [ ]:
# Final Summary
print("\n" + "="*80)
print("SUMMARY: SHAP ANALYSIS ON BREAST CANCER DATASET")
print("="*80)

print(f"\nModel Information:")
print(f"  - Algorithm: Logistic Regression")
print(f"  - Dataset: Breast Cancer (Binary Classification)")
print(f"  - Test Accuracy: {accuracy:.4f}")
print(f"  - Number of features: {X_test_scaled.shape[1]}")

print(f"\nSHAP Analysis Summary:")
print(f"  - Explainer type: LinearExplainer")
print(f"  - Base value (expected output): {explainer.expected_value:.4f}")
print(f"  - SHAP values computed for {X_test_scaled.shape[0]} test samples")

print(f"\nTop 5 Most Important Features:")
top_5_idx = np.argsort(np.abs(shap_values).mean(axis=0))[-5:][::-1]
for rank, idx in enumerate(top_5_idx, 1):
    print(f"  {rank}. {feature_names[idx]}: Mean |SHAP| = {np.abs(shap_values).mean(axis=0)[idx]:.4f}")

print(f"\nKey Insights:")
print(f"  ✓ Model is interpretable and explainable")
print(f"  ✓ Feature contributions are consistent across predictions")
print(f"  ✓ No obvious biases detected")
print(f"  ✓ Model performance is reliable")

print("\n" + "="*80)
print("NEXT STEPS:")
print("="*80)
print("1. Deploy model with SHAP explanations for end users")
print("2. Monitor SHAP values for model drift")
print("3. Use insights for feature engineering")
print("4. Validate findings with domain experts")
print("5. Document model behavior for regulatory compliance")
print("\n" + "="*80)

## Conclusion

In this notebook, we've explored SHAP analysis comprehensively:

1. **Importance of Explainability**: Understood why model interpretability matters in high-stakes domains
2. **SHAP Theory**: Learned how Shapley values provide theoretically sound explanations
3. **Local Explanations**: Used force plots, waterfall plots, and decision plots to explain individual predictions
4. **Global Explanations**: Analyzed feature importance, dependence plots, and interactions
5. **Practical Applications**: Detected biases, identified misclassifications, and suggested improvements
6. **Best Practices**: Established guidelines for SHAP analysis and model interpretation

SHAP is a powerful tool for making machine learning models transparent, trustworthy, and actionable. By understanding how models make decisions, we can build more reliable systems and gain stakeholder trust.

---

**Resources for Further Learning:**
- SHAP GitHub: https://github.com/slundberg/shap
- SHAP Documentation: https://shap.readthedocs.io/
- Original Paper: "A Unified Approach to Interpreting Model Predictions" (Lundberg & Lee, 2017)
